# Unidad 1: Arquitecturas Big Data y ETL batch distribuido

## Sesión 2: Fundamentos PySpark
### Transformaciones, funciones, agrupaciones y evaluación perezosa

### Objetivo
Aprender a transformar datos con PySpark, aplicar funciones,
agrupar registros y comprender cuándo Spark ejecuta los cálculos.

### Datos de la práctica guiada
Utilizaremos los archivos H&M indicados por el docente:
- articles.csv
- customers.csv
- transactions.parquet

### Forma de trabajo
Cada actividad incluirá una explicación, el código ejecutado
y una interpretación del resultado observado.

## 3.2 Crear la SparkSession

### ¿Qué haremos?
Crearemos la sesión que permite utilizar Spark desde Python.

### Configuración indicada por el docente
- appName: sesion2-fundamentos-spark, para identificar la aplicación.
- master: local[*], para utilizar los núcleos disponibles
  dentro del contenedor.
- spark.ui.port: 4040, accesible en Windows mediante
  http://localhost:4042.
- spark.sql.shuffle.partitions: 8, para configurar las particiones
  utilizadas al redistribuir datos en operaciones como agrupaciones.
- spark.driver.memory: 4g, para solicitar 4 GB de memoria
  para el driver.

### Resultado esperado
Una SparkSession activa con el nombre de la sesión 2.
Todavía no cargaremos los archivos H&M.

In [1]:
from pyspark.sql import SparkSession

spark = (
    SparkSession.builder
    .appName("sesion2-fundamentos-spark")
    .master("local[*]")
    .config("spark.ui.port", "4040")
    .config("spark.sql.shuffle.partitions", "8")
    .config("spark.driver.memory", "4g")
    .getOrCreate()
)

spark

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/11/16 19:51:39 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


### Definir las rutas de trabajo

Guardaremos las rutas en variables para reutilizarlas
sin escribirlas completas en cada operación.

- ORIGEN_DATOS: ubicación de los archivos H&M.
- ARTIFACTS: ubicación destinada a las salidas de la práctica.

Estas rutas corresponden al interior del contenedor.

In [2]:
ORIGEN_DATOS = "/opt/s02-fundamentos/data"
ARTIFACTS = "/opt/s02-fundamentos/artifacts"

## 3.3 Cargar y explorar articles.csv

### ¿Qué haremos?
Leeremos el archivo articles.csv, que contiene información
de los productos del dataset H&M.

### ¿Cómo lo leeremos?
- header=True: utiliza la primera fila como nombres de columnas.
- inferSchema=True: permite que Spark deduzca los tipos de datos.

### Resultado esperado
Un DataFrame llamado df_articles, preparado para explorar
sus filas y su estructura.

In [3]:
df_articles = spark.read.csv(
    f"{ORIGEN_DATOS}/articles.csv",
    header=True,
    inferSchema=True,
)

### Explorar las primeras filas

Mostraremos cinco registros del catálogo de productos.

Usaremos truncate=False para que Spark muestre los textos
completos, sin acortarlos.

In [4]:
df_articles.show(5, truncate=False)


+----------+------------+-----------------+---------------+-----------------+------------------+-----------------------+-------------------------+-----------------+-----------------+-------------------------+---------------------------+--------------------------+----------------------------+-------------+---------------+----------+----------------+--------------+----------------+----------+----------------------+----------------+------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|article_id|product_code|prod_name        |product_type_no|product_type_name|product_group_name|graphical_appearance_no|graphical_appearance_name|colour_group_code|colour_group_name|perceived_colour_value_id|perceived_colour_value_name|perceived_colour_master_id|perceived_colour_master_name|department_no|

### Revisar la estructura del DataFrame

Utilizaremos printSchema() para identificar:
- El nombre de cada columna.
- Su tipo de dato.
- Si permite valores nulos.

Esto permite comprobar cómo Spark interpretó el archivo
antes de realizar transformaciones.

In [5]:
df_articles.printSchema()

root
 |-- article_id: integer (nullable = true)
 |-- product_code: integer (nullable = true)
 |-- prod_name: string (nullable = true)
 |-- product_type_no: integer (nullable = true)
 |-- product_type_name: string (nullable = true)
 |-- product_group_name: string (nullable = true)
 |-- graphical_appearance_no: integer (nullable = true)
 |-- graphical_appearance_name: string (nullable = true)
 |-- colour_group_code: integer (nullable = true)
 |-- colour_group_name: string (nullable = true)
 |-- perceived_colour_value_id: integer (nullable = true)
 |-- perceived_colour_value_name: string (nullable = true)
 |-- perceived_colour_master_id: integer (nullable = true)
 |-- perceived_colour_master_name: string (nullable = true)
 |-- department_no: integer (nullable = true)
 |-- department_name: string (nullable = true)
 |-- index_code: string (nullable = true)
 |-- index_name: string (nullable = true)
 |-- index_group_no: integer (nullable = true)
 |-- index_group_name: string (nullable = true)

### Obtener un resumen de columnas seleccionadas

Seleccionaremos cinco columnas para que el resumen sea legible.

describe() calcula la cantidad de valores no nulos, promedio,
desviación estándar, mínimo y máximo cuando corresponde.

Las columnas department_no y section_no representan códigos:
sus promedios no deben interpretarse como medidas del negocio.

In [6]:
df_articles.select(
    "prod_name",
    "product_group_name",
    "colour_group_name",
    "department_no",
    "section_no"
).describe().show()

26/11/16 19:56:37 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.
[Stage 3:======>                                                    (1 + 8) / 9]

+-------+--------------------+------------------+-----------------+------------------+-----------------+
|summary|           prod_name|product_group_name|colour_group_name|     department_no|       section_no|
+-------+--------------------+------------------+-----------------+------------------+-----------------+
|  count|              105542|            105542|           105542|            105542|           105542|
|   mean|                NULL|              NULL|             NULL|  4532.77783252165|42.66421898391162|
| stddev|                NULL|              NULL|             NULL|2712.6920114304908|23.26010495890632|
|    min|& Denim Boyfriend...|       Accessories|            Beige|              1201|                2|
|    max|           Åsa Dress|           Unknown|  Yellowish Brown|              9989|               97|
+-------+--------------------+------------------+-----------------+------------------+-----------------+



### Visualizar registros en formato vertical

Mostraremos tres productos con sus campos organizados
verticalmente, en lugar de utilizar una tabla ancha.

Este formato facilita revisar registros con muchas columnas.

In [7]:
df_articles.show(3, vertical=True)


-RECORD 0--------------------------------------------
 article_id                   | 108775015            
 product_code                 | 108775               
 prod_name                    | Strap top            
 product_type_no              | 253                  
 product_type_name            | Vest top             
 product_group_name           | Garment Upper body   
 graphical_appearance_no      | 1010016              
 graphical_appearance_name    | Solid                
 colour_group_code            | 9                    
 colour_group_name            | Black                
 perceived_colour_value_id    | 4                    
 perceived_colour_value_name  | Dark                 
 perceived_colour_master_id   | 5                    
 perceived_colour_master_name | Black                
 department_no                | 1676                 
 department_name              | Jersey Basic         
 index_code                   | A                    
 index_name                 

## 3.4 Cargar customers.csv con esquema explícito

### ¿Qué haremos?
Definiremos la estructura de los datos de clientes antes
de leer el archivo.

### ¿Qué es un esquema explícito?
Es una definición de los nombres y tipos de las columnas.
Spark utilizará esa definición en lugar de deducir los tipos.

### ¿Para qué sirve?
Permite controlar cómo se interpretan los datos y conservar
como texto los identificadores que no deben tratarse
como números.

In [9]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType,
    IntegerType
)

schema_customers = StructType([
    StructField("customer_id", StringType(), nullable=True),
    StructField("FN", DoubleType(), nullable=True),
    StructField("Active", DoubleType(), nullable=True),
    StructField("club_member_status", StringType(), nullable=True),
    StructField("fashion_news_frequency", StringType(), nullable=True),
    StructField("age", IntegerType(), nullable=True),
    StructField("postal_code", StringType(), nullable=True),
])

### Leer el archivo de clientes

Cargaremos customers.csv utilizando schema_customers.

header=True indica que el archivo tiene encabezados.
schema=schema_customers establece los tipos de las columnas,
por lo que no necesitamos inferSchema=True.

El resultado se guardará en df_customers.

In [10]:
df_customers = spark.read.csv(
    f"{ORIGEN_DATOS}/customers.csv",
    header=True,
    schema=schema_customers
)

### Verificar el esquema de clientes

Revisaremos los nombres y tipos de las columnas para comprobar
que Spark utilizó el esquema explícito.

En particular, customer_id y postal_code deben ser texto,
mientras que age debe ser un número entero.

In [11]:
df_customers.printSchema()

root
 |-- customer_id: string (nullable = true)
 |-- FN: double (nullable = true)
 |-- Active: double (nullable = true)
 |-- club_member_status: string (nullable = true)
 |-- fashion_news_frequency: string (nullable = true)
 |-- age: integer (nullable = true)
 |-- postal_code: string (nullable = true)



### Explorar los primeros clientes

Mostraremos cinco filas de df_customers para observar
los valores de las columnas.

Utilizaremos truncate=False para mostrar completos
los identificadores y códigos postales.

In [12]:
df_customers.show(5, truncate=False)


+----------------------------------------------------------------+----+------+------------------+----------------------+---+----------------------------------------------------------------+
|customer_id                                                     |FN  |Active|club_member_status|fashion_news_frequency|age|postal_code                                                     |
+----------------------------------------------------------------+----+------+------------------+----------------------+---+----------------------------------------------------------------+
|00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|NULL|NULL  |ACTIVE            |NONE                  |49 |52043ee2162cf5aa7ee79974281641c6f11a68d276429a91f8ca0d4b6efa8100|
|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|NULL|NULL  |ACTIVE            |NONE                  |25 |2973abc54daa8a5f8ccfe9362140c63247c5eee03f1d93f4c830291c32bc3057|
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8f

### Obtener el resumen estadístico de clientes

Utilizaremos describe() para explorar los datos.

La fila count muestra cuántos valores no nulos hay en cada
columna. Si una columna tiene un conteo menor que customer_id,
puede contener valores faltantes.

Revisaremos especialmente las columnas FN y Active.


In [13]:
df_customers.describe().show()


[Stage 8:===========>                                             (4 + 16) / 20]

+-------+--------------------+------+------+------------------+----------------------+------------------+--------------------+
|summary|         customer_id|    FN|Active|club_member_status|fashion_news_frequency|               age|         postal_code|
+-------+--------------------+------+------+------------------+----------------------+------------------+--------------------+
|  count|             1371980|476930|464404|           1365918|               1355971|           1356119|             1371980|
|   mean|                NULL|   1.0|   1.0|              NULL|                  NULL|   36.386964565794|                NULL|
| stddev|                NULL|   0.0|   0.0|              NULL|                  NULL|14.313627981628093|                NULL|
|    min|00000dbacae5abe5e...|   1.0|   1.0|            ACTIVE|               Monthly|                16|0000198d2c593b7d3...|
|    max|ffffd9ac14e899464...|   1.0|   1.0|        PRE-CREATE|             Regularly|                99|ffffe1

### Consultar los nombres de las columnas

Mostraremos la lista de columnas de df_customers.

Esto permite conocer sus nombres exactos para utilizarlos
después en selecciones, filtros y agrupaciones.


In [14]:
print(df_customers.columns)

['customer_id', 'FN', 'Active', 'club_member_status', 'fashion_news_frequency', 'age', 'postal_code']


### Consultar el tamaño del DataFrame

Calcularemos la cantidad total de filas y columnas.

- count() cuenta las filas del DataFrame.
- len(df_customers.columns) cuenta sus columnas.

El conteo de filas es una acción: Spark ejecutará el trabajo
necesario para obtener el resultado.

In [15]:
num_rows, num_cols = df_customers.count(), len(df_customers.columns)

print(f"Filas: {num_rows}, Columnas: {num_cols}")

Filas: 1371980, Columnas: 7


### Crear una muestra aleatoria de clientes

Utilizaremos una muestra pequeña para explorar los datos
y probar operaciones con menos registros.

Seleccionaremos aproximadamente el 1 % de las filas,
sin reemplazo y sin una semilla fija.

La muestra se guardará en df_customers_muestra.
El DataFrame original df_customers se conserva.

In [16]:
df_customers_muestra = df_customers.sample(
    withReplacement=False,
    fraction=0.01,
    seed=None,
)

### Consultar registros finales de la muestra

Utilizaremos tail(3) para obtener tres registros del final
del recorrido actual del DataFrame.

El resultado será una lista de objetos Row.

Un DataFrame no tiene un orden garantizado si no lo ordenamos
explícitamente. Por eso, estas filas no representan
necesariamente los últimos clientes registrados.


In [18]:
df_customers_muestra.tail(3)

[Row(customer_id='fff052a0464cf292d6b451da9dba86a299c52698a3b8c338acb089a89e337a62', FN=1.0, Active=1.0, club_member_status='ACTIVE', fashion_news_frequency='Regularly', age=81, postal_code='a2cf6b5b16e6e25f61a1e8a75e66c12a1f20777fdb169f0ba062330784292bc6'),
 Row(customer_id='fff3921d91ecdb98b73a68cd6ddaae3b07751e8772fea51455e72ee95b3d8333', FN=None, Active=None, club_member_status='ACTIVE', fashion_news_frequency='NONE', age=47, postal_code='f1a22db37a6d6c837be8a24a48eb2f87a43235c2b265d66350c8846c310540ee'),
 Row(customer_id='fff782a90757aaa4c019a12d5b7567ea2a533d7cffbab324ca5ab614b5262df9', FN=1.0, Active=1.0, club_member_status='ACTIVE', fashion_news_frequency='Regularly', age=26, postal_code='05c845b77df23bcb826ab03b23929b0acd0595668aeeac37b13192f6f5da7faf')]

### Guardar la muestra en CSV y Parquet

Guardaremos df_customers_muestra en dos formatos:

- CSV: formato de texto que organiza los datos por filas.
- Parquet: formato columnar utilizado para análisis de datos.

Spark creará una carpeta para cada salida, con archivos
de datos en su interior.

El modo overwrite reemplaza la carpeta de salida si ya existe.
No modifica los archivos originales de H&M.


In [19]:
df_customers_muestra.write.mode("overwrite").csv(
    f"{ARTIFACTS}/customers_muestra_csv",
    header=True
)

df_customers_muestra.write.mode("overwrite").parquet(
    f"{ARTIFACTS}/customers_muestra_parquet"
)

### Consultar el número de particiones

Una partición es una parte de los datos que Spark puede
procesar como una tarea.

La muestra conserva las particiones de su origen, aunque
contenga menos filas. Al escribirla, Spark puede generar
varios archivos de datos dentro de la carpeta de salida.

In [20]:
df_customers_muestra.rdd.getNumPartitions()

20

### Guardar la muestra con una sola partición

Utilizaremos coalesce(1) para reducir la muestra a una partición
antes de guardarla en CSV y Parquet.

Spark seguirá creando una carpeta para cada salida,
pero contendrá un único archivo de datos part-...,
además de archivos de control como _SUCCESS.

Esta opción se utiliza aquí porque trabajamos con una
muestra pequeña.

In [21]:
df_customers_muestra.coalesce(1).write.mode("overwrite").csv(
    f"{ARTIFACTS}/customers_muestra_csv_unico",
    header=True
)

df_customers_muestra.coalesce(1).write.mode("overwrite").parquet(
    f"{ARTIFACTS}/customers_muestra_parquet_unico"
)

### Guardar un CSV con nombre exacto

Convertiremos la muestra pequeña a pandas para generar
un archivo llamado customers_muestra.csv.

Utilizaremos index=False para evitar que pandas agregue
una columna con el índice de las filas.

Esta conversión solo se utiliza cuando los datos caben
en la memoria del driver.

In [22]:
df_customers_muestra.toPandas().to_csv(
    f"{ARTIFACTS}/customers_muestra.csv",
    index=False
)


/usr/local/lib/python3.10/site-packages/pyspark/sql/pandas/conversion.py:348: UserWarning: toPandas attempted Arrow optimization because 'spark.sql.execution.arrow.pyspark.enabled' is set to true; however, failed by the reason below:
  [PACKAGE_NOT_INSTALLED] PyArrow >= 18.0.0 must be installed; however, it was not found.
Attempting non-optimization as 'spark.sql.execution.arrow.pyspark.fallback.enabled' is set to true.
  warn(msg)


### Leer la muestra guardada en Parquet

Leeremos la carpeta customers_muestra_parquet y mostraremos
cinco registros para comprobar su lectura.

Spark interpreta los archivos de datos de la carpeta
como un solo DataFrame. Parquet conserva los nombres
y tipos de las columnas.

In [23]:
df_muestra_parquet = spark.read.parquet(
    f"{ARTIFACTS}/customers_muestra_parquet"
)

df_muestra_parquet.show(5, truncate=False)

+----------------------------------------------------------------+----+------+------------------+----------------------+---+----------------------------------------------------------------+
|customer_id                                                     |FN  |Active|club_member_status|fashion_news_frequency|age|postal_code                                                     |
+----------------------------------------------------------------+----+------+------------------+----------------------+---+----------------------------------------------------------------+
|4e8a18f5baba50103aec36231d4ae323ab8476377c249ccbcba8f71839d83125|NULL|NULL  |ACTIVE            |NONE                  |66 |d6a3514e9ee2d09b7eeb331307ea066c00b984c24a5424772fd4433249242add|
|4e8e1d2b817f78feabe6996d54cd36ab6a67fa25630b1a458d8636c9cf5975c3|NULL|NULL  |ACTIVE            |NONE                  |50 |abc09e18a4d7ce3d26cb2d0a4b35b70d2adbec332d29bb9aae4251d262aa6082|
|4e8e6e5925e44d260bc0f1b2de316a0afcc0fb2f754755d46

### Comparar las salidas CSV y Parquet

Leeremos ambas carpetas y contaremos sus registros para
comprobar que contienen la misma cantidad de filas.

Esta comparación verifica la cantidad de registros,
no la igualdad completa de sus valores o tipos.

In [24]:
df_leido_csv = spark.read.csv(
    f"{ARTIFACTS}/customers_muestra_csv",
    header=True,
    inferSchema=True
)

df_leido_parquet = spark.read.parquet(
    f"{ARTIFACTS}/customers_muestra_parquet"
)

df_leido_csv.count(), df_leido_parquet.count()

(13793, 13793)

## 3.5 Transformaciones y evaluación perezosa

### Seleccionar columnas

Crearemos df_seleccionado con cuatro columnas de df_customers:
customer_id, age, club_member_status y fashion_news_frequency.

select() es una transformación: prepara un nuevo plan de
procesamiento sin calcular todavía las filas del resultado.

El DataFrame original conserva todas sus columnas.

In [25]:
df_seleccionado = df_customers.select(
    "customer_id",
    "age",
    "club_member_status",
    "fashion_news_frequency"
)

### Filtrar clientes con membresía activa

Crearemos df_activos conservando únicamente las filas
donde club_member_status sea ACTIVE.

filter() es una transformación: añade una condición
al plan, sin calcular todavía las filas del resultado.

In [26]:
from pyspark.sql.functions import col

df_activos = df_seleccionado.filter(
    col("club_member_status") == "ACTIVE"
)


### Encadenar transformaciones

Combinaremos select() y filter() para preparar el mismo
resultado en una sola expresión.

Spark conserva estas operaciones en un plan. Las filas
se calcularán cuando solicitemos una acción.


In [27]:
df_activos = (
    df_customers
    .select(
        "customer_id",
        "age",
        "club_member_status",
        "fashion_news_frequency"
    )
    .filter(col("club_member_status") == "ACTIVE")
)

df_activos

DataFrame[customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string]

### Ejecutar una acción: mostrar clientes activos

Utilizaremos show() para solicitar diez filas del resultado.

Hasta ahora select() y filter() prepararon el plan.
Esta acción hace que Spark ejecute el trabajo necesario
para obtener y mostrar las filas solicitadas.

In [28]:
df_activos.show(10, truncate=False)

+----------------------------------------------------------------+---+------------------+----------------------+
|customer_id                                                     |age|club_member_status|fashion_news_frequency|
+----------------------------------------------------------------+---+------------------+----------------------+
|00000dbacae5abe5e23885899a1fa44253a17956c6d1c3d25f88aa139fdfc657|49 |ACTIVE            |NONE                  |
|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|25 |ACTIVE            |NONE                  |
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|24 |ACTIVE            |NONE                  |
|00005ca1c9ed5f5146b52ac8639a40ca9d57aeff4d1bd2c5feb1ca5dff07c43e|54 |ACTIVE            |NONE                  |
|00006413d8573cd20ed7128e53b7b13819fe5cfc2d801fe7fc0f26dd8d65a85a|52 |ACTIVE            |Regularly             |
|0000757967448a6cb83efb3ea7a3fb9d418ac7adf2379d8cd0c725276a467a2a|20 |ACTIVE            |NONE   

### Ejecutar una acción: contar clientes activos

Utilizaremos count() para obtener la cantidad total
de filas que cumplen el filtro de membresía ACTIVE.

A diferencia de show(10), que muestra diez registros,
count() calcula el total del resultado filtrado.

In [29]:
df_activos.count()

1272491

## 3.6 Analizar el plan de ejecución

Utilizaremos explain(True) para revisar cómo Spark interpreta
y organiza las transformaciones de df_activos.

Compararemos el plan inicial con el optimizado para identificar
cambios realizados por Catalyst, el optimizador de Spark SQL.

Esta instrucción muestra el plan; no devuelve las filas
ni ejecuta su procesamiento completo.


In [30]:
df_activos.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Analyzed Logical Plan ==
customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string
Filter (club_member_status#600 = ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Optimized Logical Plan ==
Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
+- Filter (isnotnull(club_member_status#600) AND (club_member_status#600 = ACTIVE))
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== 

## 3.7 Aplicar funciones y crear columnas

Utilizaremos withColumn() para crear o reemplazar columnas
del DataFrame de artículos.

Emplearemos estas funciones:
- col(): hace referencia a una columna.
- when(): establece una condición.
- lit(): incorpora un valor constante.
- current_date(): obtiene la fecha actual.

También utilizaremos cast() para cambiar el tipo de una columna.

In [31]:
from pyspark.sql.functions import col, when, lit, current_date

### Revisar el identificador del artículo

article_id identifica un producto; no representa una cantidad.

Como leímos el archivo con inferSchema=True, comprobaremos
si Spark interpretó este identificador como un número.
Los identificadores deben conservar su formato original,
incluidos los ceros iniciales.

In [32]:
df_articles.select("article_id").printSchema()

df_articles.select("article_id").show(5)

root
 |-- article_id: integer (nullable = true)

+----------+
|article_id|
+----------+
| 108775015|
| 108775044|
| 108775051|
| 110065001|
| 110065002|
+----------+
only showing top 5 rows


### Convertir article_id a texto

Utilizaremos withColumn() para reemplazar article_id
por una versión de tipo string.

Esto permite tratarlo como un identificador de texto.
La conversión no recupera ceros iniciales que se hayan
perdido durante la lectura con inferSchema=True.

### Convertir article_id a texto

Utilizaremos withColumn() para reemplazar article_id
por una versión de tipo string.

Esto permite tratarlo como un identificador de texto.
La conversión no recupera ceros iniciales que se hayan
perdido durante la lectura con inferSchema=True.


In [34]:
df_articles = df_articles.withColumn(
    "article_id",
    col("article_id").cast("string")
)

df_articles.select("article_id").printSchema()

root
 |-- article_id: string (nullable = true)



### Clasificar el tono percibido del producto

Crearemos la columna rango_percibido a partir de
perceived_colour_value_name:

- Dark se clasificará como oscuro.
- Light se clasificará como claro.
- Cualquier otro valor, incluidos los nulos, se clasificará
  como medio, siguiendo la regla del ejercicio.
  

In [35]:
df_articles = df_articles.withColumn(
    "rango_percibido",
    when(col("perceived_colour_value_name") == "Dark", "oscuro")
    .when(col("perceived_colour_value_name") == "Light", "claro")
    .otherwise("medio")
)

### Agregar la fuente de los datos

Crearemos una columna llamada fuente con el valor
"H&M Kaggle" en todas las filas.

Utilizaremos lit() para incorporar ese texto como
un valor constante, no como el nombre de una columna.

In [36]:
df_articles = df_articles.withColumn(
    "fuente",
    lit("H&M Kaggle")
)

### Agregar la fecha de procesamiento

Crearemos la columna fecha_procesado utilizando current_date().

Esta fecha corresponde a la ejecución de la consulta en Spark,
según la zona horaria de su sesión. No representa la fecha
de creación del producto ni la fecha de una compra.

In [37]:
df_articles = df_articles.withColumn(
    "fecha_procesado",
    current_date()
)

### Verificar las columnas transformadas

Mostraremos cinco productos para revisar:
- article_id convertido a texto.
- rango_percibido según la regla de clasificación.
- fuente con el valor H&M Kaggle.
- fecha_procesado calculada por Spark.

Convertir article_id a texto no recupera los ceros iniciales
que pudieran haberse perdido durante la lectura.

In [38]:
df_articles.select(
    "article_id",
    "prod_name",
    "perceived_colour_value_name",
    "rango_percibido",
    "fuente",
    "fecha_procesado"
).show(5, truncate=False)

+----------+-----------------+---------------------------+---------------+----------+---------------+
|article_id|prod_name        |perceived_colour_value_name|rango_percibido|fuente    |fecha_procesado|
+----------+-----------------+---------------------------+---------------+----------+---------------+
|108775015 |Strap top        |Dark                       |oscuro         |H&M Kaggle|2026-11-16     |
|108775044 |Strap top        |Light                      |claro          |H&M Kaggle|2026-11-16     |
|108775051 |Strap top (1)    |Dusty Light                |medio          |H&M Kaggle|2026-11-16     |
|110065001 |OP T-shirt (Idro)|Dark                       |oscuro         |H&M Kaggle|2026-11-16     |
|110065002 |OP T-shirt (Idro)|Light                      |claro          |H&M Kaggle|2026-11-16     |
+----------+-----------------+---------------------------+---------------+----------+---------------+
only showing top 5 rows


## 3.8 Cargar transacciones y aplicar funciones

### Leer los datos

Cargaremos transactions.parquet y limitaremos el DataFrame
a un máximo de 100 000 registros para la práctica.

Parquet conserva el esquema de los datos, por lo que no
necesitamos header=True ni inferSchema=True.

Como no aplicamos un orden, el subconjunto no representa
necesariamente las primeras compras por fecha ni una
muestra aleatoria.

In [39]:
df_transactions = spark.read.parquet(
    f"{ORIGEN_DATOS}/transactions.parquet"
).limit(100000)

### Revisar el esquema de las transacciones

Utilizaremos printSchema() para conocer los nombres y tipos
de las columnas de df_transactions.

Comprobaremos la estructura real antes de transformar
fechas, identificadores y valores numéricos.

In [40]:
df_transactions.printSchema()

root
 |-- t_dat: string (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- article_id: long (nullable = true)
 |-- price: double (nullable = true)
 |-- sales_channel_id: long (nullable = true)



### Establecer los tipos de las transacciones

Aplicaremos los siguientes tipos:
- t_dat: fecha.
- customer_id y article_id: texto.
- price: número decimal.
- sales_channel_id: número entero.

Estas transformaciones no modifican el archivo original.
Convertir identificadores a texto no recupera ceros
iniciales que ya se hayan perdido.

In [41]:
from pyspark.sql.functions import col, to_date

df_transactions = (
    df_transactions
    .withColumn("t_dat", to_date(col("t_dat"), "yyyy-MM-dd"))
    .withColumn("customer_id", col("customer_id").cast("string"))
    .withColumn("article_id", col("article_id").cast("string"))
    .withColumn("price", col("price").cast("double"))
    .withColumn("sales_channel_id", col("sales_channel_id").cast("int"))
)

### Clasificar por canal de venta

Crearemos la columna canal a partir de sales_channel_id.

Para reproducir el ejercicio de la guía utilizaremos:
- 1: Online.
- 2: Tienda.
- Otros valores o nulos: Desconocido.

La correspondencia de los códigos debe verificarse con
la documentación del dataset antes de interpretar
resultados de negocio.


In [42]:
from pyspark.sql.functions import when

df_transactions = df_transactions.withColumn(
    "canal",
    when(col("sales_channel_id") == 1, "Online")
    .when(col("sales_channel_id") == 2, "Tienda")
    .otherwise("Desconocido")
)


### Clasificar el precio normalizado

Crearemos la columna categoria_precio con estas reglas:
- Menor que 0.1: Barato.
- Desde 0.1 hasta menos de 0.3: Medio.
- Desde 0.3: Caro.

Estas etiquetas corresponden al ejercicio.
price contiene valores normalizados, no importes en moneda.

Con esta expresión, los valores nulos también recibirían
la etiqueta Caro; es una limitación que debemos revisar.

In [43]:
df_transactions = df_transactions.withColumn(
    "categoria_precio",
    when(col("price") < 0.1, "Barato")
    .when(col("price") < 0.3, "Medio")
    .otherwise("Caro")
)

### Clasificar el tipo de transacción

Crearemos tipo_transaccion combinando el código del canal
y el precio normalizado.

Aplicaremos las etiquetas del ejercicio:
- Canal 1 y price mayor que 0.3: Online Premium.
- Canal 2 y price mayor que 0.3: Tienda Premium.
- Los demás casos: Regular.

Premium es una etiqueta definida para esta práctica,
no una categoría original del dataset.

In [44]:
df_transactions = df_transactions.withColumn(
    "tipo_transaccion",
    when(
        (col("sales_channel_id") == 1) & (col("price") > 0.3),
        "Online Premium"
    )
    .when(
        (col("sales_channel_id") == 2) & (col("price") > 0.3),
        "Tienda Premium"
    )
    .otherwise("Regular")
)

### Obtener el día de la semana

Crearemos dia_semana a partir de la fecha t_dat.

La función dayofweek() utiliza esta numeración:
1 = domingo, 2 = lunes, 3 = martes, 4 = miércoles,
5 = jueves, 6 = viernes y 7 = sábado.

Después utilizaremos este número para distinguir
fines de semana de lunes a viernes.

In [45]:
from pyspark.sql.functions import dayofweek

df_transactions = df_transactions.withColumn(
    "dia_semana",
    dayofweek(col("t_dat"))
)

### Clasificar el tipo de día

Crearemos tipo_dia utilizando dia_semana:
- Domingo (1) y sábado (7): Fin de semana.
- Los demás valores: Laborable.

La etiqueta Laborable representa lunes a viernes;
no considera feriados. Siguiendo esta expresión,
los valores nulos también quedarían como Laborable.

In [46]:
df_transactions = df_transactions.withColumn(
    "tipo_dia",
    when(
        col("dia_semana").isin(1, 7),
        "Fin de semana"
    ).otherwise("Laborable")
)

## 3.9 Agrupaciones y agregaciones

### Sumar el precio normalizado por cliente

Agruparemos las transacciones por customer_id y sumaremos
price dentro de cada grupo.

El resultado tendrá una fila por cliente presente
en el subconjunto de transacciones utilizado.

price está normalizado: la suma no representa dinero
en soles, dólares ni otra moneda.

In [47]:
from pyspark.sql.functions import sum

df_total_por_cliente = df_transactions.groupBy("customer_id").agg(
    sum("price").alias("total_normalizado")
)

df_total_por_cliente.show(5, truncate=False)

[Stage 39:==>                                                     (1 + 19) / 20]

+----------------------------------------------------------------+------------------+
|customer_id                                                     |total_normalizado |
+----------------------------------------------------------------+------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|0.081322033898305 |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|0.0863559322033897|
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|0.190593220338983 |
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|0.0428474576271185|
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|0.7130508474576257|
+----------------------------------------------------------------+------------------+
only showing top 5 rows


### Calcular el promedio por cliente

Agruparemos las transacciones por customer_id y calcularemos
el promedio de price para cada cliente.

El promedio corresponde al precio normalizado de sus
transacciones dentro del subconjunto utilizado.
No representa un importe monetario.

In [48]:
from pyspark.sql.functions import avg

df_avg_por_cliente = df_transactions.groupBy("customer_id").agg(
    avg("price").alias("promedio_normalizado")
)

df_avg_por_cliente.show(5, truncate=False)

[Stage 42:===============================================>        (17 + 3) / 20]

+----------------------------------------------------------------+--------------------+
|customer_id                                                     |promedio_normalizado|
+----------------------------------------------------------------+--------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|0.0406610169491525  |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|0.01727118644067794 |
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|0.038118644067796595|
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|0.02142372881355925 |
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|0.023768361581920857|
+----------------------------------------------------------------+--------------------+
only showing top 5 rows


### Contar transacciones por cliente

Agruparemos los registros por customer_id y contaremos
las filas de cada grupo.

count("*") incluye todas las filas, aunque alguna
de sus columnas contenga valores nulos.

El resultado corresponde al subconjunto de transacciones
utilizado, no al historial completo.

In [49]:
from pyspark.sql.functions import count

df_count_por_cliente = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones")
)

df_count_por_cliente.show(5, truncate=False)

[Stage 45:============================================>           (16 + 4) / 20]

+----------------------------------------------------------------+-----------------+
|customer_id                                                     |num_transacciones|
+----------------------------------------------------------------+-----------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|2                |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|5                |
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|5                |
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|2                |
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|30               |
+----------------------------------------------------------------+-----------------+
only showing top 5 rows


### Contar clientes únicos por día

Agruparemos las transacciones por fecha y contaremos
los identificadores de cliente distintos.

Si un cliente tiene varias transacciones el mismo día,
se contará una sola vez para esa fecha.

El cálculo corresponde al subconjunto utilizado.

In [50]:
from pyspark.sql.functions import countDistinct

df_clientes_unicos_por_dia = df_transactions.groupBy("t_dat").agg(
    countDistinct("customer_id").alias("clientes_unicos")
)

df_clientes_unicos_por_dia.show(5)

[Stage 48:=====>                                                  (2 + 18) / 20]

+----------+---------------+
|     t_dat|clientes_unicos|
+----------+---------------+
|2018-09-20|          13987|
|2018-09-21|          13932|
|2018-09-22|           1285|
+----------+---------------+



### Combinar varias agregaciones

Calcularemos para cada cliente:
- Número de transacciones.
- Suma del precio normalizado.
- Promedio del precio normalizado.

Utilizaremos una sola agrupación para reunir los tres
resultados en la misma tabla.

Los valores corresponden al subconjunto utilizado
y no representan importes en moneda.

In [51]:
from pyspark.sql.functions import sum, count, avg

df_agg_multi = df_transactions.groupBy("customer_id").agg(
    count("*").alias("num_transacciones"),
    sum("price").alias("total_normalizado"),
    avg("price").alias("promedio_normalizado")
)

df_agg_multi.show(5, truncate=False)

[Stage 51:>                                                       (0 + 20) / 20]

+----------------------------------------------------------------+-----------------+------------------+--------------------+
|customer_id                                                     |num_transacciones|total_normalizado |promedio_normalizado|
+----------------------------------------------------------------+-----------------+------------------+--------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|2                |0.081322033898305 |0.0406610169491525  |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|5                |0.0863559322033897|0.01727118644067794 |
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|5                |0.190593220338983 |0.038118644067796595|
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|2                |0.0428474576271185|0.02142372881355925 |
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|30               |0.7130508474576257|0.023768361581920857|


### Agrupar por fecha y canal de venta

Calcularemos la suma de price para cada combinación
de fecha y código de canal.

Una misma fecha puede aparecer en varias filas,
una por cada canal presente.

La suma corresponde a precios normalizados
del subconjunto utilizado, no a importes en moneda.

In [52]:
df_ventas_por_dia_y_canal = df_transactions.groupBy(
    "t_dat",
    "sales_channel_id"
).agg(
    sum("price").alias("total_normalizado")
)

df_ventas_por_dia_y_canal.show(5)

+----------+----------------+------------------+
|     t_dat|sales_channel_id| total_normalizado|
+----------+----------------+------------------+
|2018-09-20|               2| 1072.852593220397|
|2018-09-20|               1|342.49325423728436|
|2018-09-21|               2|1036.5931525424558|
|2018-09-21|               1| 382.6691525423702|
|2018-09-22|               1|55.550881355931914|
+----------+----------------+------------------+
only showing top 5 rows


### Aplicar agregaciones mediante un diccionario

Agruparemos por cliente y utilizaremos un diccionario
para indicar dos cálculos:
- Sumar price.
- Contar los valores no nulos de article_id.

Después cambiaremos los nombres automáticos de las columnas
por nombres más claros.

La suma de price corresponde a valores normalizados,
no a importes monetarios.

In [53]:
df_agg_dict = (
    df_transactions.groupBy("customer_id")
    .agg({"price": "sum", "article_id": "count"})
    .withColumnRenamed("sum(price)", "total_normalizado")
    .withColumnRenamed("count(article_id)", "num_articulos")
)

df_agg_dict.show(5, truncate=False)

[Stage 57:>                                                       (0 + 20) / 20]

+----------------------------------------------------------------+-------------+------------------+
|customer_id                                                     |num_articulos|total_normalizado |
+----------------------------------------------------------------+-------------+------------------+
|000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318|2            |0.081322033898305 |
|00007d2de826758b65a93dd24ce629ed66842531df6699338c5570910a014cc2|5            |0.0863559322033897|
|00083cda041544b2fbb0e0d2905ad17da7cf1007526fb4c73235dccbbc132280|5            |0.190593220338983 |
|0008968c0d451dbc5a9968da03196fe20051965edde7413775c4eb3be9abe9c2|2            |0.0428474576271185|
|000aa7f0dc06cd7174389e76c9e132a67860c5f65f970699daccc14425ac31a8|30           |0.7130508474576257|
+----------------------------------------------------------------+-------------+------------------+
only showing top 5 rows


### Definir una ventana por cliente

Utilizaremos Window.partitionBy("customer_id") para indicar
que los cálculos se realizarán entre las transacciones
del mismo cliente.

A diferencia de groupBy().agg(), una función ventana
permite conservar las filas individuales y añadirles
un resultado calculado para su grupo.

In [54]:
from pyspark.sql.window import Window

window_cliente = Window.partitionBy("customer_id")

### Añadir el total del cliente a cada transacción

Calcularemos la suma de price para cada cliente mediante
la ventana definida anteriormente.

Cada transacción conservará su fila y tendrá una columna
adicional con el total normalizado de su cliente.

El total corresponde únicamente al subconjunto utilizado.

In [55]:
from pyspark.sql.functions import sum

df_con_total_cliente = df_transactions.withColumn(
    "total_gastado_cliente",
    sum("price").over(window_cliente)
)

df_con_total_cliente.show(5, truncate=False)

[Stage 60:=====>                                                  (2 + 18) / 20]

+----------+----------------------------------------------------------------+----------+------------------+----------------+------+----------------+----------------+----------+---------+---------------------+
|t_dat     |customer_id                                                     |article_id|price             |sales_channel_id|canal |categoria_precio|tipo_transaccion|dia_semana|tipo_dia |total_gastado_cliente|
+----------+----------------------------------------------------------------+----------+------------------+----------------+------+----------------+----------------+----------+---------+---------------------+
|2018-09-21|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|583558001 |0.0677796610169491|2               |Tienda|Barato          |Regular         |6         |Laborable|0.12706779661016931  |
|2018-09-21|0000423b00ade91418cceaf3b26c6af3dd342b51fd051eec9c12fb36984420fa|639677008 |0.0254067796610169|2               |Tienda|Barato          |Regular         

## 3.10 Procesar texto mediante RDD

### Obtener las descripciones de productos

Seleccionaremos detail_desc de df_articles y convertiremos
sus filas en un RDD de textos.

Eliminaremos las descripciones nulas antes de procesarlas.
Después mostraremos cinco ejemplos.

Un RDD es una colección distribuida que permite aplicar
operaciones directamente sobre sus elementos.

In [56]:
rdd = df_articles.select("detail_desc").rdd.map(
    lambda x: x.detail_desc
)

rdd = rdd.filter(lambda texto: texto is not None)

rdd.take(5)

['Jersey top with narrow shoulder straps.',
 'Jersey top with narrow shoulder straps.',
 'Jersey top with narrow shoulder straps.',
 'Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.',
 'Microfibre T-shirt bra with underwired, moulded, lightly padded cups that shape the bust and provide good support. Narrow adjustable shoulder straps and a narrow hook-and-eye fastening at the back. Without visible seams for greater comfort.']

### Filtrar descripciones que mencionan algodón

Buscaremos el texto cotton dentro de las descripciones.

Convertiremos cada descripción a minúsculas para encontrar
coincidencias sin distinguir entre mayúsculas y minúsculas.

Mostraremos cinco resultados.

In [57]:
textos_algodon = rdd.filter(
    lambda texto: "cotton" in texto.lower()
)

textos_algodon.take(5)

['Sweatshirt in soft organic cotton with a  press-stud on one shoulder (sizes 12-18 months and 18-24 months without a press-stud). Brushed inside.',
 'Sweatshirt in soft organic cotton with a  press-stud on one shoulder (sizes 12-18 months and 18-24 months without a press-stud). Brushed inside.',
 'Leggings in soft jersey with a wide panel at the waist for best fit over the tummy. The cotton content of the leggings is organic.',
 'Leggings in soft jersey with a wide panel at the waist for best fit over the tummy. The cotton content of the leggings is organic.',
 'Leggings in soft jersey with a wide panel at the waist for best fit over the tummy. The cotton content of the leggings is organic.']

### Contar palabras de las descripciones

Separaremos las descripciones en palabras y contaremos
cuántas veces aparece cada una.

Primero convertiremos los textos a minúsculas y sustituiremos
signos de puntuación por espacios. Después asignaremos
el valor 1 a cada palabra y sumaremos sus apariciones.

Mostraremos las diez palabras más frecuentes.

In [58]:
import re
from operator import add

palabras = rdd.flatMap(
    lambda linea: re.sub(
        r"[^\wáéíóúñüÁÉÍÓÚÑÜ]", " ", linea.lower()
    ).split()
)

pares = palabras.filter(
    lambda p: p != ""
).map(
    lambda palabra: (palabra, 1)
)

conteo = pares.reduceByKey(add)

conteo.takeOrdered(10, key=lambda x: -x[1])

[('and', 163384),
 ('a', 152403),
 ('with', 150704),
 ('the', 136096),
 ('in', 109801),
 ('at', 80690),
 ('back', 36827),
 ('front', 36316),
 ('soft', 35611),
 ('waist', 34416)]

### Reto 1: contar las apariciones de cotton

Consultaremos el conteo distribuido para obtener
la frecuencia de la palabra cotton.

El resultado representa apariciones de la palabra,
no necesariamente la cantidad de productos que la mencionan.


In [59]:
conteo.filter(
    lambda x: x[0] == "cotton"
).collect()

[('cotton', 34348)]

### Reto 2: buscar descripciones con sustainable

Filtraremos las descripciones que contienen el texto
sustainable y mostraremos hasta cinco resultados.

Esta búsqueda identifica una mención en la descripción;
no verifica por sí sola la sostenibilidad del producto.

In [60]:
rdd.filter(
    lambda texto: "sustainable" in texto.lower()
).take(5)

['CONSCIOUS EXCLUSIVE. Short Bohemian top in a jacquard-weave Orange Fiber™ and Tencel™ lyocell blend with a pattern in glittery threads. Sweetheart neckline at the top with a silicone trim and integral support panels and grosgrain trims that fasten on the inside for extra support, and a zip at the back. Pleats front and back and 3/4-length balloon sleeves with narrow elastication at the top and cuffs. Organic cotton lining. Orange Fiber™ is made from citrus peel that is a by-product of juice production transformed into a sustainable, high-quality fabric that reduces waste and saves natural resources.',
 'CONSCIOUS EXCLUSIVE. Top in matt satin made from a silk and Tencel™ lyocell blend with wide, tie-top shoulder straps in a contrasting colour and a V-neck. Opening with a covered button at the back and slits in the sides. Lined at the top. The top is dyed in a natural, sustainable way using coffee.',
 'Studio Collection. 70s-inspired, 5-pocket jeans in unwashed cotton denim. Fitted at 

In [61]:
    conteo.takeOrdered(10, key=lambda x: -x[1])

[('and', 163384),
 ('a', 152403),
 ('with', 150704),
 ('the', 136096),
 ('in', 109801),
 ('at', 80690),
 ('back', 36827),
 ('front', 36316),
 ('soft', 35611),
 ('waist', 34416)]

### Reto 3: interpretar las palabras más frecuentes

Seis de las diez palabras más frecuentes son stopwords: and, a, with, the, in y at.
El vocabulario del catálogo comienza en la posición 7 con back; después aparecen front, soft y waist.
Las palabras comunes dominan el conteo porque no se aplicó un filtro de stopwords.


## 3.11 Hallazgos y reflexión sobre la práctica

### Extracción y exploración

Cargué articles.csv con inferencia de tipos y customers.csv
con un esquema explícito.

En articles.csv revisé las filas, el esquema y un resumen
de columnas seleccionadas. En customers.csv consulté
sus columnas, cantidad de registros y estadísticas.

#### Resultados de mi ejecución
- Cantidad de clientes: 1 371 980.
- Cantidad de columnas de clientes: 7.
- Valores no nulos en FN: 476 930.
- Valores no nulos en Active: 464 404.

#### Interpretación
Un conteo de valores no nulos menor que la cantidad total
de filas indica datos faltantes en esa columna.

### Transformaciones y evaluación perezosa

Utilicé select() para elegir columnas y filter() para
conservar clientes con membresía ACTIVE.

Estas transformaciones prepararon el plan de procesamiento.
Al ejecutar show() y count(), solicité resultados y Spark
ejecutó el trabajo necesario para obtenerlos.

También utilicé withColumn() para convertir tipos y crear
columnas mediante condiciones, valores constantes y fechas.

#### Resultados de mi ejecución
- Cantidad de clientes con membresía ACTIVE: 1 272 491.
- Ejemplo de columna creada: rango_percibido.
- Regla aplicada: Dark se convierte en oscuro,
  Light en claro y los demás casos en medio.

#### Limitaciones identificadas
Convertir article_id a string no recupera ceros iniciales
que se hayan perdido al leerlo como número.

Las reglas con otherwise() pueden asignar una categoría
a valores nulos, por lo que deben revisarse antes de
utilizarse en un análisis real.

In [62]:
df_activos.count()

1272491

### Agrupaciones y agregaciones

Agrupé las transacciones por cliente para calcular:
- Cantidad de transacciones.
- Suma de price.
- Promedio de price.

También conté clientes distintos por fecha y calculé
la suma de price por combinación de fecha y canal.

Estos resultados corresponden al subconjunto de hasta
100 000 transacciones utilizado durante la práctica.

#### Interpretación de price

price contiene valores normalizados. Sus sumas y promedios
no representan importes en soles, dólares ni otra moneda.

#### Diferencia entre agrupación y ventana

groupBy().agg() genera una fila de resumen por grupo.

La función ventana conserva cada transacción y añade
el total de su cliente mediante:
sum("price").over(window_cliente).

Ese total se repite en las filas del mismo cliente
y no debe sumarse nuevamente para calcular un total general.

#### Ejemplo de mi ejecución

En df_agg_multi observé:
- customer_id: 000058a12d5b43e67d225668fa1f8d618c13dc232df0cad8ffe7ad4a1091e318.
- num_transacciones: 2.
- total_normalizado: 0.081322033898305.
- promedio_normalizado: 0.0406610169491525.

Estos resultados corresponden al subconjunto de transacciones
utilizado. Los valores normalizados no representan dinero
en una moneda.

### Procesamiento de texto mediante RDD

Convertí detail_desc de df_articles en un RDD de textos
y eliminé las descripciones nulas.

Utilicé estas operaciones:
- map(): extraer textos y crear pares (palabra, 1).
- filter(): seleccionar descripciones o palabras.
- flatMap(): separar las descripciones en palabras.
- reduceByKey(): sumar las apariciones de cada palabra.

El conteo incluye descripciones repetidas del catálogo;
no representa necesariamente textos únicos.

#### Resultados de mi ejecución

La palabra cotton aparece 34 348 veces.

Las diez palabras más frecuentes fueron:
1. and: 163 384.
2. a: 152 403.
3. with: 150 704.
4. the: 136 096.
5. in: 109 801.
6. at: 80 690.
7. back: 36 827.
8. front: 36 316.
9. soft: 35 611.
10. waist: 34 416.

Las primeras seis son palabras funcionales del inglés.
El vocabulario relacionado con las prendas comienza
en la posición 7, con back.

#### Interpretación

El conteo permite identificar palabras frecuentes,
pero las palabras comunes del idioma pueden dominar
el resultado si no se aplica un filtro de stopwords.

Encontrar sustainable en una descripción solo demuestra
que el texto contiene esa expresión; no verifica
la sostenibilidad del producto.
La consulta con sustainable devolvió tres descripciones. Es la cantidad de ejemplos recuperados por take(5), no un conteo calculado por separado.


In [63]:
df_activos.explain(True)

== Parsed Logical Plan ==
'Filter '`=`('club_member_status, ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Analyzed Logical Plan ==
customer_id: string, age: int, club_member_status: string, fashion_news_frequency: string
Filter (club_member_status#600 = ACTIVE)
+- Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== Optimized Logical Plan ==
Project [customer_id#597, age#602, club_member_status#600, fashion_news_frequency#601]
+- Filter (isnotnull(club_member_status#600) AND (club_member_status#600 = ACTIVE))
   +- Relation [customer_id#597,FN#598,Active#599,club_member_status#600,fashion_news_frequency#601,age#602,postal_code#603] csv

== 

### Interpretación del plan de ejecución

#### Plan inicial y analizado

El plan inicial representa la selección de cuatro columnas
seguida del filtro de membresía ACTIVE.

En el plan analizado, Spark reconoce las columnas
y establece sus tipos.

#### Plan optimizado

Catalyst coloca Filter antes de Project en el recorrido
de los datos. El plan se lee desde abajo hacia arriba:
origen → filtro → selección.

También añade isnotnull(club_member_status), porque una
membresía nula no puede cumplir la condición igual a ACTIVE.

#### Plan físico

PushedFilters muestra:
- IsNotNull(club_member_status).
- EqualTo(club_member_status, ACTIVE).

Esto confirma que Spark entrega esas condiciones
al lector CSV.

ReadSchema contiene únicamente las cuatro columnas
necesarias: customer_id, club_member_status,
fashion_news_frequency y age.

#### Optimización identificada

Se observa el desplazamiento del filtro hacia el origen
y la reducción de las columnas requeridas.

Al tratarse de CSV, esto no significa que Spark pueda
evitar recorrer el archivo como lo haría con ciertos
formatos columnares o datos particionados.

## 4. Actividad autónoma: AgroSmart Puno

### Propósito

Aplicar el flujo de transformación de PySpark a datos
agroclimáticos correspondientes a Azángaro, Puno.

### Dimensiones del proyecto

- Atmósfera: temperatura del aire y humedad relativa.
- Suelo: temperatura y humedad del suelo, según disponibilidad.

### Actividades requeridas

1. Cargar los datos y explorar su estructura.
2. Aplicar al menos tres transformaciones y una acción.
3. Interpretar el plan de ejecución con explain().
4. Aplicar funciones de columna.
5. Realizar una agrupación y una agregación relevante.
6. Convertir a RDD y aplicar al menos dos operaciones.

### Procedencia y alcance

Documentaremos la ubicación, el periodo, las unidades
y la fuente de los datos utilizados.

Los datos meteorológicos de modelos y reanálisis no
equivalen a mediciones de sensores de una parcela.
El análisis identificará condiciones de frío; no confirmará
daños en cultivos.

### Descargar los datos históricos

Enviaremos a Open-Meteo la consulta preparada para Azángaro,
correspondiente a junio de 2025.

La respuesta se recibirá en formato JSON, que organiza
los datos mediante campos y listas.

Primero comprobaremos que la consulta responda correctamente.
Después guardaremos la respuesta y la prepararemos para Spark.

In [66]:
import json
from urllib.parse import urlencode
from urllib.request import urlopen

URL_API = "https://archive-api.open-meteo.com/v1/archive"

PARAMETROS = {
    "latitude": -14.91,
    "longitude": -70.20,
    "start_date": "2025-06-01",
    "end_date": "2025-06-30",
    "hourly": ",".join([
        "temperature_2m",
        "relative_humidity_2m",
        "soil_temperature_0_to_7cm",
        "soil_moisture_0_to_7cm"
    ]),
    "timezone": "America/Lima"
}

url_consulta = f"{URL_API}?{urlencode(PARAMETROS)}"

with urlopen(url_consulta, timeout=60) as respuesta:
    datos_api = json.load(respuesta)

print("Campos recibidos:", list(datos_api.keys()))
print("Variables horarias:", list(datos_api.get("hourly", {}).keys()))

Campos recibidos: ['latitude', 'longitude', 'generationtime_ms', 'utc_offset_seconds', 'timezone', 'timezone_abbreviation', 'elevation', 'hourly_units', 'hourly']
Variables horarias: ['time', 'temperature_2m', 'relative_humidity_2m', 'soil_temperature_0_to_7cm', 'soil_moisture_0_to_7cm']


### Verificar la respuesta de la API

Revisaremos la cantidad de registros horarios, el periodo,
las unidades y las coordenadas devueltas.

Las coordenadas devueltas pueden diferir del punto solicitado
porque los datos corresponden a una celda del modelo.
La humedad del suelo no debe interpretarse como porcentaje
sin comprobar su unidad.

In [67]:
datos_horarios = datos_api["hourly"]

print("Coordenadas devueltas:",
      datos_api["latitude"], datos_api["longitude"])
print("Zona horaria:", datos_api["timezone"])
print("Unidades:", datos_api["hourly_units"])

for variable, valores in datos_horarios.items():
    print(f"{variable}: {len(valores)} registros")

print("Primera fecha:", datos_horarios["time"][0])
print("Última fecha:", datos_horarios["time"][-1])

Coordenadas devueltas: -14.938488 -70.186584
Zona horaria: America/Lima
Unidades: {'time': 'iso8601', 'temperature_2m': '°C', 'relative_humidity_2m': '%', 'soil_temperature_0_to_7cm': '°C', 'soil_moisture_0_to_7cm': 'm³/m³'}
time: 720 registros
temperature_2m: 720 registros
relative_humidity_2m: 720 registros
soil_temperature_0_to_7cm: 720 registros
soil_moisture_0_to_7cm: 720 registros
Primera fecha: 2025-06-01T00:00
Última fecha: 2025-06-30T23:00


### Conservar los datos originales y su procedencia

Guardaremos la respuesta de Open-Meteo en formato JSON.

También registraremos la consulta y la fecha de descarga.
Esto permite identificar de dónde proceden los datos
y distinguir su periodo histórico del momento de obtención.

Si el archivo ya existe, se utilizará otro nombre
para conservar la descarga anterior.

In [68]:
from pathlib import Path
from datetime import datetime, timezone

carpeta_agro = Path(ORIGEN_DATOS) / "agrosmart"
carpeta_agro.mkdir(parents=True, exist_ok=True)

fecha_descarga = datetime.now(timezone.utc)
marca = fecha_descarga.strftime("%Y%m%dT%H%M%S%fZ")

ruta_original = carpeta_agro / f"open_meteo_azangaro_{marca}.json"

registro_descarga = {
    "fuente": "Open-Meteo Historical Weather API",
    "fecha_descarga_utc": fecha_descarga.isoformat(),
    "url_consulta": url_consulta,
    "parametros": PARAMETROS,
    "respuesta": datos_api
}

with ruta_original.open("x", encoding="utf-8") as archivo:
    json.dump(
        registro_descarga,
        archivo,
        ensure_ascii=False,
        indent=2
    )

print("Archivo guardado:", ruta_original)

Archivo guardado: /opt/s02-fundamentos/data/agrosmart/open_meteo_azangaro_20261116T210745703464Z.json


### Organizar los datos horarios en registros

La API entrega una lista por variable. Cada posición
corresponde a la misma hora en todas las listas.

Uniremos esos valores para obtener una fila por hora,
con temperatura y humedad del aire y del suelo.

Conservaremos las unidades indicadas por la API.
La humedad del suelo se mantendrá en m³/m³.

In [69]:
registros_agro = []

for i, fecha_hora in enumerate(datos_horarios["time"]):
    registros_agro.append({
        "fecha_hora": fecha_hora,
        "localidad": "Azangaro",
        "temperatura_aire_c": datos_horarios["temperature_2m"][i],
        "humedad_relativa_pct": datos_horarios["relative_humidity_2m"][i],
        "temperatura_suelo_c": datos_horarios["soil_temperature_0_to_7cm"][i],
        "humedad_suelo_m3_m3": datos_horarios["soil_moisture_0_to_7cm"][i]
    })

print("Registros preparados:", len(registros_agro))
print("Primer registro:", registros_agro[0])

Registros preparados: 720
Primer registro: {'fecha_hora': '2025-06-01T00:00', 'localidad': 'Azangaro', 'temperatura_aire_c': 2.9, 'humedad_relativa_pct': 70, 'temperatura_suelo_c': 4.3, 'humedad_suelo_m3_m3': 0.212}


### Crear el DataFrame de AgroSmart Puno

Definiremos los nombres y tipos de las columnas antes
de convertir los registros a un DataFrame de Spark.

La fecha se cargará inicialmente como texto y después
se convertirá a timestamp.

Las variables ambientales utilizarán números decimales
y permitirán valores nulos.

In [70]:
from pyspark.sql.types import (
    StructType,
    StructField,
    StringType,
    DoubleType
)

schema_agro = StructType([
    StructField("fecha_hora", StringType(), True),
    StructField("localidad", StringType(), True),
    StructField("temperatura_aire_c", DoubleType(), True),
    StructField("humedad_relativa_pct", DoubleType(), True),
    StructField("temperatura_suelo_c", DoubleType(), True),
    StructField("humedad_suelo_m3_m3", DoubleType(), True),
])

registros_spark = [
    {
        campo: (
            float(valor)
            if campo in {
                "temperatura_aire_c",
                "humedad_relativa_pct",
                "temperatura_suelo_c",
                "humedad_suelo_m3_m3"
            } and valor is not None
            else valor
        )
        for campo, valor in registro.items()
    }
    for registro in registros_agro
]

df_agro = spark.createDataFrame(
    registros_spark,
    schema=schema_agro
)

### Explorar el DataFrame de AgroSmart Puno

Revisaremos los nombres y tipos de las columnas mediante
printSchema().

Después mostraremos cinco registros para comprobar
las fechas, la localidad y las variables ambientales.

In [71]:
df_agro.printSchema()

df_agro.show(5, truncate=False)

root
 |-- fecha_hora: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- temperatura_aire_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- temperatura_suelo_c: double (nullable = true)
 |-- humedad_suelo_m3_m3: double (nullable = true)

+----------------+---------+------------------+--------------------+-------------------+-------------------+
|fecha_hora      |localidad|temperatura_aire_c|humedad_relativa_pct|temperatura_suelo_c|humedad_suelo_m3_m3|
+----------------+---------+------------------+--------------------+-------------------+-------------------+
|2025-06-01T00:00|Azangaro |2.9               |70.0                |4.3                |0.212              |
|2025-06-01T01:00|Azangaro |2.9               |70.0                |4.0                |0.212              |
|2025-06-01T02:00|Azangaro |2.8               |70.0                |3.8                |0.212              |
|2025-06-01T03:00|Azangaro |2.1               |74.0  

### Convertir la fecha y hora

La API entrega fecha_hora como texto en la zona America/Lima.

Configuraremos esa misma zona en Spark y crearemos
fecha_hora_ts como timestamp. Esto permitirá realizar
operaciones por fecha y hora.

Conservaremos fecha_hora para comparar el dato original
con el resultado convertido.

In [72]:
from pyspark.sql.functions import col, to_timestamp

spark.conf.set("spark.sql.session.timeZone", "America/Lima")

df_agro_transformado = df_agro.withColumn(
    "fecha_hora_ts",
    to_timestamp(col("fecha_hora"), "yyyy-MM-dd'T'HH:mm")
)

### Obtener la fecha de cada registro

Crearemos la columna fecha a partir de fecha_hora_ts.

Esta columna conservará únicamente el año, mes y día,
y permitirá agrupar las observaciones por fecha local
de America/Lima.

In [73]:
from pyspark.sql.functions import to_date

df_agro_transformado = df_agro_transformado.withColumn(
    "fecha",
    to_date(col("fecha_hora_ts"))
)

### Clasificar la temperatura del aire

Crearemos condicion_temperatura con estas reglas:
- Temperatura nula: Sin dato.
- Temperatura menor o igual a 0 °C: A cero o bajo cero.
- Temperatura mayor que 0 °C: Sobre cero.

Esta clasificación describe la temperatura del aire del modelo.
No confirma una helada en la parcela ni daño en un cultivo.

In [74]:
from pyspark.sql.functions import when

df_agro_transformado = df_agro_transformado.withColumn(
    "condicion_temperatura",
    when(col("temperatura_aire_c").isNull(), "Sin dato")
    .when(col("temperatura_aire_c") <= 0, "A cero o bajo cero")
    .otherwise("Sobre cero")
)

### Ejecutar y revisar las transformaciones

Hasta este punto preparamos las columnas fecha_hora_ts,
fecha y condicion_temperatura.

Utilizaremos show() para solicitar filas del resultado
y ejecutar el procesamiento necesario para mostrarlas.

In [75]:
df_agro_transformado.select(
    "fecha_hora",
    "fecha_hora_ts",
    "fecha",
    "temperatura_aire_c",
    "condicion_temperatura"
).show(10, truncate=False)

+----------------+-------------------+----------+------------------+---------------------+
|fecha_hora      |fecha_hora_ts      |fecha     |temperatura_aire_c|condicion_temperatura|
+----------------+-------------------+----------+------------------+---------------------+
|2025-06-01T00:00|2025-06-01 00:00:00|2025-06-01|2.9               |Sobre cero           |
|2025-06-01T01:00|2025-06-01 01:00:00|2025-06-01|2.9               |Sobre cero           |
|2025-06-01T02:00|2025-06-01 02:00:00|2025-06-01|2.8               |Sobre cero           |
|2025-06-01T03:00|2025-06-01 03:00:00|2025-06-01|2.1               |Sobre cero           |
|2025-06-01T04:00|2025-06-01 04:00:00|2025-06-01|1.9               |Sobre cero           |
|2025-06-01T05:00|2025-06-01 05:00:00|2025-06-01|2.1               |Sobre cero           |
|2025-06-01T06:00|2025-06-01 06:00:00|2025-06-01|1.8               |Sobre cero           |
|2025-06-01T07:00|2025-06-01 07:00:00|2025-06-01|3.9               |Sobre cero           |

### Analizar el plan de ejecución del proyecto

Utilizaremos explain(True) para revisar cómo Spark organiza
las transformaciones de fecha y clasificación de temperatura.

Compararemos el plan inicial con el optimizado para identificar
una optimización real, sin asumir que será igual a la del CSV
de clientes.

In [77]:
df_agro_transformado.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(condicion_temperatura, CASE WHEN 'isNull('temperatura_aire_c) THEN Sin dato WHEN '`<=`('temperatura_aire_c, 0) THEN A cero o bajo cero ELSE Sobre cero END, None)]
+- Project [fecha_hora#1532, localidad#1533, temperatura_aire_c#1534, humedad_relativa_pct#1535, temperatura_suelo_c#1536, humedad_suelo_m3_m3#1537, fecha_hora_ts#1557, to_date(fecha_hora_ts#1557, None, Some(America/Lima), true) AS fecha#1558]
   +- Project [fecha_hora#1532, localidad#1533, temperatura_aire_c#1534, humedad_relativa_pct#1535, temperatura_suelo_c#1536, humedad_suelo_m3_m3#1537, to_timestamp(fecha_hora#1532, Some(yyyy-MM-dd'T'HH:mm), TimestampType, Some(America/Lima), true) AS fecha_hora_ts#1557]
      +- LogicalRDD [fecha_hora#1532, localidad#1533, temperatura_aire_c#1534, humedad_relativa_pct#1535, temperatura_suelo_c#1536, humedad_suelo_m3_m3#1537], false

== Analyzed Logical Plan ==
fecha_hora: string, localidad: string, temperatura_aire_c: double

### Calcular indicadores diarios

Agruparemos los registros por localidad y fecha para calcular:
- Temperatura mínima del aire.
- Temperatura promedio del aire.
- Humedad relativa promedio.
- Cantidad de registros con temperatura disponible.
- Cantidad de registros horarios con temperatura a cero
  o bajo cero.

Los valores nulos no se contarán como temperaturas positivas.
El resultado describe datos del modelo, no daños en cultivos.

In [78]:
from pyspark.sql import functions as F

df_resumen_diario = df_agro_transformado.groupBy(
    "localidad", "fecha"
).agg(
    F.min("temperatura_aire_c").alias("temperatura_minima_c"),
    F.avg("temperatura_aire_c").alias("temperatura_promedio_c"),
    F.avg("humedad_relativa_pct").alias("humedad_relativa_promedio_pct"),
    F.count("temperatura_aire_c").alias("registros_con_temperatura"),
    F.sum(
        F.when(F.col("temperatura_aire_c") <= 0, 1).otherwise(0)
    ).alias("registros_a_cero_o_bajo_cero")
)

df_resumen_diario.orderBy("fecha").show(10, truncate=False)

+---------+----------+--------------------+----------------------+-----------------------------+-------------------------+----------------------------+
|localidad|fecha     |temperatura_minima_c|temperatura_promedio_c|humedad_relativa_promedio_pct|registros_con_temperatura|registros_a_cero_o_bajo_cero|
+---------+----------+--------------------+----------------------+-----------------------------+-------------------------+----------------------------+
|Azangaro |2025-06-01|1.8                 |6.966666666666668     |56.916666666666664           |24                       |0                           |
|Azangaro |2025-06-02|-0.4                |6.224999999999999     |59.916666666666664           |24                       |2                           |
|Azangaro |2025-06-03|-1.6                |6.279166666666666     |49.541666666666664           |24                       |4                           |
|Azangaro |2025-06-04|-2.3                |6.279166666666668     |54.125                

### Integrar atmósfera y suelo en el resumen diario

Ampliaremos el resumen con:
- Temperatura mínima del suelo.
- Humedad promedio del suelo.
- Cantidad de registros con humedad del suelo disponible.

Las variables del suelo corresponden a la profundidad
de 0 a 7 cm. La humedad se expresa en m³/m³,
no en porcentaje.

In [79]:
df_resumen_diario = df_agro_transformado.groupBy(
    "localidad", "fecha"
).agg(
    F.min("temperatura_aire_c").alias("temperatura_minima_aire_c"),
    F.avg("temperatura_aire_c").alias("temperatura_promedio_aire_c"),
    F.avg("humedad_relativa_pct").alias("humedad_relativa_promedio_pct"),
    F.count("temperatura_aire_c").alias("registros_con_temperatura_aire"),
    F.sum(
        F.when(F.col("temperatura_aire_c") <= 0, 1).otherwise(0)
    ).alias("registros_a_cero_o_bajo_cero"),
    F.min("temperatura_suelo_c").alias("temperatura_minima_suelo_c"),
    F.avg("humedad_suelo_m3_m3").alias("humedad_promedio_suelo_m3_m3"),
    F.count("humedad_suelo_m3_m3").alias("registros_con_humedad_suelo")
)

df_resumen_diario.orderBy("fecha").show(10, truncate=False)

+---------+----------+-------------------------+---------------------------+-----------------------------+------------------------------+----------------------------+--------------------------+----------------------------+---------------------------+
|localidad|fecha     |temperatura_minima_aire_c|temperatura_promedio_aire_c|humedad_relativa_promedio_pct|registros_con_temperatura_aire|registros_a_cero_o_bajo_cero|temperatura_minima_suelo_c|humedad_promedio_suelo_m3_m3|registros_con_humedad_suelo|
+---------+----------+-------------------------+---------------------------+-----------------------------+------------------------------+----------------------------+--------------------------+----------------------------+---------------------------+
|Azangaro |2025-06-01|1.8                      |6.966666666666668          |56.916666666666664           |24                            |0                           |3.4                       |0.21037500000000006         |24                       

### Procesar registros de frío mediante RDD

Convertiremos las columnas fecha y temperatura del aire
a un RDD.

Aplicaremos:
- filter(): conservar temperaturas disponibles menores
  o iguales a 0 °C.
- map(): crear pares (fecha, 1).
- reduceByKey(): sumar los registros por fecha.

Las fechas sin registros que cumplan la condición
no aparecerán en este resultado.

In [80]:
from operator import add

rdd_agro = df_agro_transformado.select(
    "fecha", "temperatura_aire_c"
).rdd

rdd_frio = rdd_agro.filter(
    lambda fila: (
        fila["fecha"] is not None
        and fila["temperatura_aire_c"] is not None
        and fila["temperatura_aire_c"] <= 0
    )
)

conteo_frio_por_dia = (
    rdd_frio
    .map(lambda fila: (str(fila["fecha"]), 1))
    .reduceByKey(add)
)

conteo_frio_por_dia.takeOrdered(10, key=lambda par: par[0])

[('2025-06-02', 2),
 ('2025-06-03', 4),
 ('2025-06-04', 4),
 ('2025-06-05', 2),
 ('2025-06-06', 6),
 ('2025-06-07', 2),
 ('2025-06-08', 3),
 ('2025-06-09', 5),
 ('2025-06-10', 1),
 ('2025-06-11', 2)]

### Interpretación de los resultados del proyecto

#### Fuente y alcance
Utilicé datos históricos de Open-Meteo para un punto
de referencia de Azángaro, del 1 al 30 de junio de 2025.

Los datos representan estimaciones de un modelo,
no mediciones de sensores instalados en una parcela.

#### Transformaciones
Convertí la fecha y hora a timestamp, extraje la fecha
y clasifiqué la temperatura del aire respecto de 0 °C.

Las transformaciones prepararon el plan. show() solicitó
la ejecución necesaria para visualizar sus resultados.

#### Agrupaciones
Calculé indicadores diarios de atmósfera y suelo.
Los conteos de valores disponibles permiten revisar
con cuántos registros se calculó cada indicador.

#### Resultado observado

Para el 2 de junio de 2025, observé:
- Temperatura mínima del aire: -0.4 °C.
- Temperatura mínima del suelo: 2.9 °C.
- Humedad promedio del suelo: aproximadamente 0.20433 m³/m³.
- Registros con aire a cero o bajo cero: 2.
- Registros disponibles de temperatura del aire: 24.

#### Procesamiento RDD

Apliqué filter(), map() y reduceByKey() para contar
los registros con aire a cero o bajo cero por fecha.

Para el 2 de junio de 2025, el resultado RDD fue:
('2025-06-02', 2).

Este conteo coincide con el resumen diario del DataFrame.
Son dos registros horarios que cumplen la condición,
no una medición de la duración exacta de una helada.


#### Límite de interpretación
Los registros a cero o bajo cero no equivalen a una
confirmación de helada en una parcela ni de daño al cultivo.

### Revisar valores faltantes

Contaremos los valores nulos de las cuatro variables
ambientales y de la fecha convertida.

Esta comprobación permitirá documentar si faltan datos.
No reemplazaremos los valores nulos por cero, porque cero
puede ser una temperatura válida.

In [81]:
columnas_revisar = [
    "fecha_hora_ts",
    "temperatura_aire_c",
    "humedad_relativa_pct",
    "temperatura_suelo_c",
    "humedad_suelo_m3_m3"
]

df_agro_transformado.agg(
    F.count("*").alias("total_registros"),
    *[
        F.sum(
            F.when(F.col(nombre).isNull(), 1).otherwise(0)
        ).alias(f"nulos_{nombre}")
        for nombre in columnas_revisar
    ]
).show(truncate=False)

+---------------+-------------------+------------------------+--------------------------+-------------------------+-------------------------+
|total_registros|nulos_fecha_hora_ts|nulos_temperatura_aire_c|nulos_humedad_relativa_pct|nulos_temperatura_suelo_c|nulos_humedad_suelo_m3_m3|
+---------------+-------------------+------------------------+--------------------------+-------------------------+-------------------------+
|720            |0                  |0                       |0                         |0                        |0                        |
+---------------+-------------------+------------------------+--------------------------+-------------------------+-------------------------+



### Reflexión técnica

La evaluación perezosa permite preparar varias transformaciones antes de ejecutarlas.  
Spark analiza el plan completo y puede reorganizar operaciones para reducir trabajo.  
Esto permite combinar pasos y limitar las columnas necesarias según la consulta.  
En AgroSmart preparé conversiones de fecha y una clasificación antes de usar show().  
La acción solicitó el procesamiento necesario para obtener el resultado.  
Si cada transformación se ejecutara inmediatamente, podrían repetirse lecturas y cálculos.  
Por eso, revisar explain() ayuda a comprender el plan y comprobar sus optimizaciones.

In [83]:
df_agro_transformado.explain(True)

== Parsed Logical Plan ==
'Project [unresolvedstarwithcolumns(condicion_temperatura, CASE WHEN 'isNull('temperatura_aire_c) THEN Sin dato WHEN '`<=`('temperatura_aire_c, 0) THEN A cero o bajo cero ELSE Sobre cero END, None)]
+- Project [fecha_hora#1532, localidad#1533, temperatura_aire_c#1534, humedad_relativa_pct#1535, temperatura_suelo_c#1536, humedad_suelo_m3_m3#1537, fecha_hora_ts#1557, to_date(fecha_hora_ts#1557, None, Some(America/Lima), true) AS fecha#1558]
   +- Project [fecha_hora#1532, localidad#1533, temperatura_aire_c#1534, humedad_relativa_pct#1535, temperatura_suelo_c#1536, humedad_suelo_m3_m3#1537, to_timestamp(fecha_hora#1532, Some(yyyy-MM-dd'T'HH:mm), TimestampType, Some(America/Lima), true) AS fecha_hora_ts#1557]
      +- LogicalRDD [fecha_hora#1532, localidad#1533, temperatura_aire_c#1534, humedad_relativa_pct#1535, temperatura_suelo_c#1536, humedad_suelo_m3_m3#1537], false

== Analyzed Logical Plan ==
fecha_hora: string, localidad: string, temperatura_aire_c: double

### Interpretación del plan de AgroSmart Puno

#### Plan analizado

El plan contiene tres operaciones Project, correspondientes
a la creación del timestamp, la fecha y la clasificación
de temperatura.

#### Optimización identificada

En el plan optimizado aparecen dos operaciones Project.
Catalyst combina la creación de fecha y condicion_temperatura
en una misma proyección.

Conserva una proyección anterior para obtener fecha_hora_ts,
porque la columna fecha depende de ese resultado.

También simplifica la conversión de fecha a:
cast(fecha_hora_ts as date).

#### Plan físico

El origen aparece como Scan ExistingRDD porque creamos
el DataFrame a partir de registros preparados en Python.

Las operaciones marcadas con *(1) pertenecen a una misma
etapa de generación de código.

No aparece un Exchange en este plan: estas transformaciones
no requieren redistribuir los registros entre particiones.

#### Conclusión

Spark reorganiza las expresiones antes de procesarlas.
La reducción de proyecciones es una optimización observable,
pero este plan por sí solo no demuestra cuánto tiempo ahorra.

In [84]:
columnas_revisar = [
    "fecha_hora_ts",
    "temperatura_aire_c",
    "humedad_relativa_pct",
    "temperatura_suelo_c",
    "humedad_suelo_m3_m3"
]

df_agro_transformado.agg(
    F.count("*").alias("total_registros"),
    *[
        F.sum(
            F.when(F.col(nombre).isNull(), 1).otherwise(0)
        ).alias(f"nulos_{nombre}")
        for nombre in columnas_revisar
    ]
).show(vertical=True)

-RECORD 0-------------------------
 total_registros            | 720 
 nulos_fecha_hora_ts        | 0   
 nulos_temperatura_aire_c   | 0   
 nulos_humedad_relativa_pct | 0   
 nulos_temperatura_suelo_c  | 0   
 nulos_humedad_suelo_m3_m3  | 0   



### Hallazgo: disponibilidad de datos agroclimáticos

#### Qué revisé
Comprobé los valores nulos de la fecha convertida y de las
cuatro variables ambientales de AgroSmart Puno.

#### Cómo lo identifiqué
Ejecuté una agregación que cuenta el total de registros
y los valores nulos de cada columna.

#### Resultado de mi ejecución
- Total de registros: 720.
- Nulos en fecha_hora_ts: 0.
- Nulos en temperatura_aire_c: 0.
- Nulos en humedad_relativa_pct: 0.
- Nulos en temperatura_suelo_c: 0.
- Nulos en humedad_suelo_m3_m3: 0.

No encontré valores faltantes en las columnas revisadas.
Esto no demuestra que todos los valores sean exactos
ni descarta duplicados u otros problemas de calidad..

#### Decisión
Conservar los valores faltantes como nulos, sin sustituirlos
por cero. Los resúmenes diarios incluyen conteos de datos
disponibles para interpretar sus resultados.

#### Limitación
Esta revisión detecta valores faltantes, pero no demuestra
que los datos sean exactos ni que estén libres de duplicados
o valores fuera de rango.

## Organización de evidencias para el informe S02

### 1. Extracción y exploración del DataFrame
- Fuente, ubicación y periodo de los datos de AgroSmart.
- Esquema y primeras filas de df_agro.
- Resultado de la revisión de valores nulos.

### 2. Transformaciones, funciones y evaluación perezosa
- Conversión de fecha_hora a timestamp.
- Creación de fecha y condicion_temperatura.
- Acción show() y explicación de cuándo se ejecuta el cálculo.

### 3. Agrupaciones y agregaciones
- Código y resultado del resumen diario.
- Interpretación de indicadores de atmósfera y suelo.
- Unidades y limitaciones del análisis.

### 4. RDD y plan de ejecución
- Operaciones filter(), map() y reduceByKey().
- Resultado del conteo de registros de frío por fecha.
- Salida de explain() y explicación de la reducción
  de tres proyecciones a dos.

### Apartados adicionales
- Datos del estudiante y enlace de GitHub.
- Hallazgo técnico.
- Reflexión técnica de 5 a 8 líneas.

In [85]:
df_agro.printSchema()

df_agro.show(5, truncate=False)

root
 |-- fecha_hora: string (nullable = true)
 |-- localidad: string (nullable = true)
 |-- temperatura_aire_c: double (nullable = true)
 |-- humedad_relativa_pct: double (nullable = true)
 |-- temperatura_suelo_c: double (nullable = true)
 |-- humedad_suelo_m3_m3: double (nullable = true)

+----------------+---------+------------------+--------------------+-------------------+-------------------+
|fecha_hora      |localidad|temperatura_aire_c|humedad_relativa_pct|temperatura_suelo_c|humedad_suelo_m3_m3|
+----------------+---------+------------------+--------------------+-------------------+-------------------+
|2025-06-01T00:00|Azangaro |2.9               |70.0                |4.3                |0.212              |
|2025-06-01T01:00|Azangaro |2.9               |70.0                |4.0                |0.212              |
|2025-06-01T02:00|Azangaro |2.8               |70.0                |3.8                |0.212              |
|2025-06-01T03:00|Azangaro |2.1               |74.0  

In [86]:
df_agro_transformado.select(
    "fecha_hora",
    "fecha_hora_ts",
    "fecha",
    "temperatura_aire_c",
    "condicion_temperatura"
).show(10, truncate=False)

+----------------+-------------------+----------+------------------+---------------------+
|fecha_hora      |fecha_hora_ts      |fecha     |temperatura_aire_c|condicion_temperatura|
+----------------+-------------------+----------+------------------+---------------------+
|2025-06-01T00:00|2025-06-01 00:00:00|2025-06-01|2.9               |Sobre cero           |
|2025-06-01T01:00|2025-06-01 01:00:00|2025-06-01|2.9               |Sobre cero           |
|2025-06-01T02:00|2025-06-01 02:00:00|2025-06-01|2.8               |Sobre cero           |
|2025-06-01T03:00|2025-06-01 03:00:00|2025-06-01|2.1               |Sobre cero           |
|2025-06-01T04:00|2025-06-01 04:00:00|2025-06-01|1.9               |Sobre cero           |
|2025-06-01T05:00|2025-06-01 05:00:00|2025-06-01|2.1               |Sobre cero           |
|2025-06-01T06:00|2025-06-01 06:00:00|2025-06-01|1.8               |Sobre cero           |
|2025-06-01T07:00|2025-06-01 07:00:00|2025-06-01|3.9               |Sobre cero           |